<a href="https://colab.research.google.com/github/catarinayee/MALDI-Assistant/blob/main/maldiast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Purpose**: This block of code installs the necessary packages for this program to run

**Input**: None :)

In [ ]:
# Install what we need.
!pip install -q numpy pandas

[**Optional**]

**Purpose**: The next block of code allows you to **merge two spreadsheets** together. Skip this if you only have one file.

**Input**: Upload **Two(2) METASPACE .CSV files** here that you just downloaded, one at a time.

In [ ]:
import sys
import os
import traceback
import numpy as np
import pandas as pd
from google.colab import files

mergefile1 = files.upload()
mergefile2 = files.upload()
mf1name = list(mergefile1.keys())[0]
mf2name = list(mergefile2.keys())[0]


mf1df = pd.read_csv(mf1name, skiprows=2)
mf2df = pd.read_csv(mf2name, skiprows=2)

mergedfile = pd.concat([mf1df, mf2df])
mergedfile.sort_values(by="mz")
mergedfile.to_csv("Merged.csv", index=False)

**Purpose**: This block of code will clean up your METASPACE file and give you a list of mz that you can copy into SCiLS.

**Input**: Did you run the [OPTIONAL] block of code right before this?
*   If you did, **no input.** Just press play.
*   If you didn't, upload the **One (1) METASPACE .CSV file** you just downloaded here.



In [ ]:
import sys
import os
import traceback
import numpy as np
import pandas as pd
from google.colab import files

if os.path.exists("Merged.csv"):
  filename="Merged.csv"

else:
  userinfile = files.upload()
  filename = list(userinfile.keys())[0]

rawpercent = 1.0
rawpercent = float(input("What raw percent threshold do you want? (i.e. 0.2) "))

if "Merged" in filename:
  infile = mergedfile
else:
  infile = pd.read_csv(filename, skiprows=2)

notdupes = infile.drop_duplicates(subset=["mz"])
noraw = notdupes[(notdupes["rawOffSampleProb"] < rawpercent) | (notdupes["rawOffSampleProb"].isna())]

noraw["mz"].to_csv("m.z. values to put into SCiLS.csv", index=False)

**Purpose**: Takes your SCiLS output to annotate and reformat the file, and then outputs an xslx file ready for fold change analysis.

**Input**: One(1) .CSV file from SCiLS.

In [ ]:
from typing_extensions import final
from datetime import datetime

scilsinfile = files.upload()
scilsfilename = list(scilsinfile.keys())[0]

infilescils = pd.read_csv(scilsfilename, sep=";", skiprows=8)
infilescils = infilescils.drop(columns=['Interval Width (+/- Da)', 'Color'])

#Prepares to align both dataframes by mz values for merging
noraw['mz'] = noraw['mz'].round(4)
infilescils = infilescils.drop(columns='Name')

finaldf = infilescils.join(noraw[['moleculeNames', 'ion', 'adduct', 'datasetName']], how='inner')

cols = finaldf.columns.tolist()
cols = cols[-4:] + cols[:-4]
finaldf = finaldf[cols]

outfiledate = datetime.today().strftime('%Y%m%d')

if "Merged" in filename:
  outfilename = f"{outfiledate}_Merged_For_Analysis_{mf1name.removesuffix(".csv")}.xlsx"
else:
  outfilename = f"{outfiledate}_For_Analysis_{filename.removesuffix(".csv")}.xlsx"

finaldf.to_excel(outfilename, index=False)
